In [1]:
import torch
from typing import Any, Callable, Dict, List, Optional, Union
from diffusers import StableDiffusionPipeline
import random
import numpy as np
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline, DDIMScheduler
from tqdm import tqdm
from torch.nn.functional import cosine_similarity
from collections import defaultdict
from sklearn.cluster import KMeans
from transformers import CLIPTokenizer, CLIPTextModel
import open_clip
from open_clip.model import CustomTextCLIP

/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
def get_esd_trainable_parameters(esd_unet, train_method='esd-x'):
    esd_params = []
    esd_param_names = []
    for name, module in esd_unet.named_modules():
        if module.__class__.__name__ in ["Linear", "Conv2d", "LoRACompatibleLinear", "LoRACompatibleConv"]:
            if train_method == 'esd-x' and 'attn2' in name:
                for n, p in module.named_parameters():
                    esd_param_names.append(name+'.'+n)
                    esd_params.append(p)
                    
            if train_method == 'esd-u' and ('attn2' not in name):
                for n, p in module.named_parameters():
                    esd_param_names.append(name+'.'+n)
                    esd_params.append(p)
                    
            if train_method == 'esd-all' :
                for n, p in module.named_parameters():
                    esd_param_names.append(name+'.'+n)
                    esd_params.append(p)
                    
            if train_method == 'esd-x-strict' and ('attn2.to_k' in name or 'attn2.to_v' in name):
                for n, p in module.named_parameters():
                    esd_param_names.append(name+'.'+n)
                    esd_params.append(p)
                    
    return esd_param_names, esd_params


In [2]:


DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
MODEL_ID = "CompVis/stable-diffusion-v1-4"

# Load your pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None
).to(DEVICE)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

unet = pipe.unet

Loading pipeline components...:  50%|████████████████████████▌                        | 3/6 [00:00<00:00,  6.79it/s]/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Loading pipeline components...: 100%|█████████████████████████████████████████████████| 6/6 [00:01<00:00,  5.99it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers tea

In [17]:
unet

UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): Conv2d(320, 320, kernel_size=(1, 1), stride=(1, 1))
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(in_features=320, out_fe

In [16]:
learnable_param_names, learnable_params = get_esd_trainable_parameters(unet, train_method='esd-x')


# print(f"Learnable parameters for ESD-X:\n {learnable_param_names}")

for name, learnable_param in zip(learnable_param_names, learnable_params):
    print(f"{name}: {learnable_param.shape}")
    


down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_q.weight: torch.Size([320, 320])
down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_k.weight: torch.Size([320, 768])
down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_v.weight: torch.Size([320, 768])
down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0.weight: torch.Size([320, 320])
down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0.bias: torch.Size([320])
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_q.weight: torch.Size([320, 320])
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_k.weight: torch.Size([320, 768])
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_v.weight: torch.Size([320, 768])
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_out.0.weight: torch.Size([320, 320])
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_out.0.bias: torch.Size([320])
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_q.weight: torch.Size([640, 640])
down_blo

['down_blocks.0.attentions.0.transformer_blocks.0.attn2.processor',
 'down_blocks.0.attentions.1.transformer_blocks.0.attn2.processor',
 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor',
 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor',
 'down_blocks.2.attentions.0.transformer_blocks.0.attn2.processor',
 'down_blocks.2.attentions.1.transformer_blocks.0.attn2.processor',
 'mid_block.attentions.0.transformer_blocks.0.attn2.processor',
 'up_blocks.1.attentions.0.transformer_blocks.0.attn2.processor',
 'up_blocks.1.attentions.1.transformer_blocks.0.attn2.processor',
 'up_blocks.1.attentions.2.transformer_blocks.0.attn2.processor',
 'up_blocks.2.attentions.0.transformer_blocks.0.attn2.processor',
 'up_blocks.2.attentions.1.transformer_blocks.0.attn2.processor',
 'up_blocks.2.attentions.2.transformer_blocks.0.attn2.processor',
 'up_blocks.3.attentions.0.transformer_blocks.0.attn2.processor',
 'up_blocks.3.attentions.1.transformer_blocks.0.attn2.processor',


In [8]:
# install hook processor for this step
hook = AttentionGradientHook()
unet.set_attn_processor(hook)

✅ Hook installed at: down_blocks.0.attentions.0.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.0.attentions.0.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.0.attentions.1.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.0.attentions.1.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.2.attentions.0.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.2.attentions.0.transformer_blocks.0.attn2.processor
✅ Hook installed at: down_blocks.2.attentions.1.transformer_blocks.0.attn1.processor
✅ Hook installed at: down_blocks.2.attentions.1.transformer_block

# batching

In [ ]:

prompts = [
    "A photo of a cat sitting on a chair",
    "A photo of a dog playing with a ball",
    "A photo of a bird flying in the sky",
    "A photo of a car driving on a road",
    "A photo of a person riding a bicycle",
]
text_embeds, _ = pipe.encode_prompt(prompt=prompts,
                                                    device='cuda:1',
                                                    num_images_per_prompt=1,
                                                    do_classifier_free_guidance=False,
                                                    negative_prompt="",
                                                    )

print(text_embeds.shape)  # should be (batchsize, 77, 768)


noise_pred_esd_model = pipe.unet(
    xt,
    timestep,
    encoder_hidden_states=erase_embeds if erase_concept_from is None else erase_from_embeds,
    timestep_cond=timestep_cond,
    cross_attention_kwargs=None,
    added_cond_kwargs=None,
    return_dict=False,
)[0]

        

torch.Size([5, 77, 768])


In [42]:
B = 4
slice_A = slice(0, B//2)      # first half = unlearn
slice_B = slice(B//2, B)      # second half = preserve

slice_A

t = torch.tensor([1,2,3,4])

t[slice_A]
t[slice_B]



tensor([3, 4])

In [46]:

batchsize = 1
sA = slice(0, (2*batchsize)//2)
sB= slice((2*batchsize)//2, 2*batchsize)

t = torch.tensor([1,2])
print(t[sA])
print(t[sB])


tensor([1])
tensor([2])
